## **Setup**

In [1]:
# Notebook autoreload
%load_ext autoreload
%autoreload 2

In [2]:
# Path setup
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve().parents[0]  # notebooks/ is one level down
sys.path.append(str(PROJECT_ROOT / "src"))

In [3]:
# Libraries
import os as os
import json as json

# Local
from chatGnT.config import CFG, ensure_dirs
from chatGnT.data import load, preprocess, tokenize

# Setup
ensure_dirs(CFG)

/Users/slacksa/miniconda3/envs/chatGnT/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## **Load & Clean Data**

In [4]:
df = load.load_all()
df_clean = preprocess.clean_recipes(df)
df_clean_filt = preprocess.filter_recipes(df_clean)

# Add text to the printed output, like "Number of recipes: XX"
print(f"Number of recipes: {df_clean['id'].nunique()}")
print(f"Number of unique ingredients: {df_clean['ingredient_name'].nunique()}")
print(f"Number of recipes after filtering: {df_clean_filt['id'].nunique()}")
print(f"Number of unique ingredients after filtering: {df_clean_filt['ingredient_name'].nunique()}")

Number of recipes: 621
Number of unique ingredients: 1614
Number of recipes after filtering: 478
Number of unique ingredients after filtering: 1108


## **Tokenize Data**

In [5]:
# Multi-task tokenization
tokens_mt = tokenize.recipe_to_tokens_mt(df_clean_filt)

# Print token example
print("Tokenized recipe example:")
print(tokens_mt[0])
print("Tokenized recipe example:")
print(tokens_mt[1])

Tokenized recipe example:
[('<amt>1 part</amt>', '<ingred>coconut-rum</ingred>'), ('<amt>0.5 part</amt>', '<ingred>amaretto</ingred>'), ('<amt>4 parts</amt>', '<ingred>orange-juice</ingred>'), ('<amt>0.5 part</amt>', '<ingred>grenadine</ingred>')]
Tokenized recipe example:
[('<amt>2 parts</amt>', '<ingred>light-rum</ingred>'), ('<amt>4 parts</amt>', '<ingred>ginger-beer</ingred>'), ('<amt>1 twist</amt>', '<ingred>lemon-peel</ingred>')]


In [6]:
# Single-task tokenization
tokens_st = tokenize.recipe_to_tokens_st(df_clean_filt)

# Print token example
print("Tokenized recipe example:")
print(tokens_st[0])
print("Tokenized recipe example:")
print(tokens_st[1])

Tokenized recipe example:
['<amt>1 part</amt>', '<ingred>coconut-rum</ingred>', '<amt>0.5 part</amt>', '<ingred>amaretto</ingred>', '<amt>4 parts</amt>', '<ingred>orange-juice</ingred>', '<amt>0.5 part</amt>', '<ingred>grenadine</ingred>']
Tokenized recipe example:
['<amt>2 parts</amt>', '<ingred>light-rum</ingred>', '<amt>4 parts</amt>', '<ingred>ginger-beer</ingred>', '<amt>1 twist</amt>', '<ingred>lemon-peel</ingred>']


In [7]:
# Save tokens to file for use in next step
with open(CFG.outputs_dir / "models/tokens_mt.json", "w") as f:
    json.dump(tokens_mt, f)

with open(CFG.outputs_dir / "models/tokens_st.json", "w") as f:
    json.dump(tokens_st, f)